## Editing efficiency, ABE focused v1 and v2

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

BASE = '/Users/kexindong/Documents/GitHub/B-ALL-in-vivo-base-editing/'

In [ ]:
FOCUSED = pd.read_csv(BASE + 'MBESv2_focused.csv')
ABE = FOCUSED[(FOCUSED['Editor'] == 'ABE') & (FOCUSED['classification'] == 'targeting guide')]

min_sensor_reads = 50

# rep_col name -> crispresso filename prefix
# v1:  d15_rep1 -> d15-rep1,  bm1 -> bm1,  spleen1 -> spleen1
# fsr: d15-1    -> d15-rep1,  bm1 -> bm1,  spleen1 -> spleen1
def rep_to_filename(rep):
    rep = rep.replace('d15_rep', 'd15-rep').replace('d15-', 'd15-rep')
    return rep + '_compact_unfiltered.csv'

def build_editing_df(crispresso_dir, rep_cols_screen, label_map, library_df, min_reads=50):
    """One data point per used replicate = average editing % across targeting guides."""
    records = []
    for cond, reps in rep_cols_screen.items():
        tissue_label = label_map[cond]
        for rep in reps:
            fname = rep_to_filename(rep)
            path = crispresso_dir + fname
            g = pd.read_csv(path).rename(columns={'Guide_ID': 'gRNA_id'})
            g2 = pd.merge(g, library_df, on='gRNA_id')
            g2 = g2[g2['Reads_aligned_all_amplicons'] >= min_reads]
            records.append({'Sample': tissue_label, 'Editing %': np.average(g2['target_base_edit_perc']), 'Edit Type': 'Target Editing (w/ Bystanders)'})
            records.append({'Sample': tissue_label, 'Editing %': np.average(g2['corr_perc']),              'Edit Type': 'Pure Correct Editing'})
    return pd.DataFrame(records)

label_map = {'d15': 'D15', 'bm': 'Bone Marrow', 'spleen': 'Spleen'}

# Only used replicates per rep_cols
rep_cols_bc_v1  = {'d15': ['d15_rep1','d15_rep2','d15_rep3'], 'bm': ['bm1','bm3','bm5'],           'spleen': ['spleen1','spleen3']}
rep_cols_bc_fsr = {'d15': ['d15-1','d15-2','d15-3'],          'bm': ['bm1','bm2','bm3','bm4','bm5'],'spleen': ['spleen1','spleen2','spleen4','spleen5']}
rep_cols_epo_v1 = {'d15': ['d15_rep1','d15_rep2','d15_rep3'], 'bm': ['bm1','bm2','bm3','bm4','bm5'],'spleen': ['spleen1','spleen2','spleen3','spleen4','spleen5']}
rep_cols_epo_fsr= {'d15': ['d15-1','d15-2','d15-3'],          'bm': ['bm1','bm2','bm3','bm4','bm5'],'spleen': ['spleen1','spleen2','spleen3','spleen4','spleen5']}

ABE_BC_V1_EDITING   = build_editing_df(BASE + 'crispresso/focused/ABE_BC/',  rep_cols_bc_v1,   label_map, ABE)
ABE_BC_FSR_EDITING  = build_editing_df(BASE + 'crispresso/fsr/ABE_BC/',      rep_cols_bc_fsr,  label_map, ABE)
ABE_EPO_V1_EDITING  = build_editing_df(BASE + 'crispresso/focused/ABE_EPO/', rep_cols_epo_v1,  label_map, ABE)
ABE_EPO_FSR_EDITING = build_editing_df(BASE + 'crispresso/fsr/ABE_EPO/',     rep_cols_epo_fsr, label_map, ABE)

print('Done loading editing data.')

## Editing efficiency barplot — 2x2

In [ ]:
order = ['D15', 'Spleen', 'Bone Marrow']

panels = [
    (0, 0, ABE_BC_V1_EDITING,   'ABE Barcoding v1'),
    (0, 1, ABE_BC_FSR_EDITING,  'ABE Barcoding v2'),
    (1, 0, ABE_EPO_V1_EDITING,  'ABE EPO v1'),
    (1, 1, ABE_EPO_FSR_EDITING, 'ABE EPO v2'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)

for row, col, df, title in panels:
    ax = axes[row][col]
    sns.barplot(data=df, y='Sample', x='Editing %', hue='Edit Type',
                ax=ax, edgecolor='black', linewidth=1,
                palette=['#3a5a40', '#a3b18a'], order=order)
    sns.stripplot(data=df, y='Sample', x='Editing %', hue='Edit Type',
                  ax=ax, edgecolor='black', linewidth=1,
                  palette=['#3a5a40', '#a3b18a'], order=order, dodge=True, s=5)
    ax.set_title(title, fontsize=16)
    ax.set_xlabel('Editing %', fontsize=16)
    ax.set_ylabel('')
    ax.set_xticks([0, 20, 40, 60, 80, 100])
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize=16)
    ax.legend([], [], frameon=False)

from matplotlib.patches import Patch
handles = [
    Patch(facecolor='#3a5a40', edgecolor='black', label='Target Editing (w/ Bystanders)'),
    Patch(facecolor='#a3b18a', edgecolor='black', label='Pure Correct Editing'),
]
fig.legend(handles=handles, fontsize=13, loc='lower center', ncol=2,
           bbox_to_anchor=(0.5, -0.03), frameon=False)

fig.tight_layout()
os.makedirs(BASE + 'figures/supp/', exist_ok=True)
fig.savefig(BASE + 'figures/supp/editing_efficiency_ABE_v1_v2.pdf', bbox_inches='tight')
plt.show()